# How the graph is built: criterion x quantile

The dial notebooks move what the receptor vector is *mixed from*. This one moves
**which edges exist at all**. The message-passing graph keeps only K train molecules:
the quantile sets how many survive, the criterion picks which ones
(`orbind/mol_selection.py`).

Every number the paper reports stands on ONE point of this grid. The figures below
exist to say whether that point is defensible, not merely inherited -- so the point
itself is marked on every panel.

**This notebook draws and does not aggregate.** Fold means, intervals and paired
differences all come from `scripts/article_sweeps/s4_quantile_grid.py`, which delegates
the averaging rule to `alpha_grid`: model seeds are averaged *inside* each fold, and
the interval is over folds. A figure that re-derived its own means would treat the
(fold, seed) cells as independent observations and halve every band on the page.

Produced by `scripts/article_sweeps/s4_run_quantile_criteria.py`.

In [ ]:
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredOffsetbox, HPacker, TextArea, VPacker

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks" / "article_figures"))

import figkit as fk
from scripts.article_sweeps import s4_quantile_grid as qg

fk.use_house_style()
print("repo:", ROOT)

## Knobs

In [ ]:
# ================================== KNOBS ====================================
ROOT_DIR   = "results/article_sweeps/quantile_criteria"
DATASET    = None          # None -> everything on disk; or "m2or" / ["cc", "hc"]
REGIME     = None          # None -> both; or "transductive" / "inductive"
MOL_SOURCE = None          # None -> whatever is there; or "chemberta"
COMBO      = "cls+mol"     # the head; the reference arm is kept whatever this says
SPLIT      = "val"         # where a quantile may be CHOSEN. Decided 25.09:
                           # this page works on validation and `test` is read
                           # ONCE, at the end, for the cell already picked

SEEDS      = None          # None -> every seed on disk. Name them to cut back to
                           # a COMPLETE grid when a seed was added and its run
                           # has not finished -- see the balance check below
CI_LEVEL   = 0.95          # Student-t over FOLDS (seeds averaged inside each fold)
METRICS    = None          # None -> the metric of record per dataset
HIGHLIGHT  = None          # None -> all eight in their own hue, which is what the
                           # palette was sized for and what a CRITERION comparison
                           # needs. "auto" dims all but the reported one -- right
                           # when the figure's job is "ours against the rest"
SHOW_K     = True          # the K panel: what the quantile actually cuts

SAVE_FIGS  = False
FIG_DIR    = "results/article_figures/quantile_criteria"
FIG_FORMATS = ("png", "pdf")   # vector for print, PNG for looking at it here

# ------------------- the headline pair (25.09), in the dial's idiom ----------
# The same knobs as `prediction_dial`'s highlight figure, and for the same reasons:
# built at the FINAL printed width, because fonts do not scale when a figure is shrunk
# into a column.
PANELS     = [("m2or", "transductive"), ("m2or", "inductive")]
FIG_WIDTH_IN = 7.1     # full page width (180 mm): two panels of 3.55 in
ASPECT     = 0.90      # height / width of one panel; two big panels can be near square
BAND_CRITS = ("greedy_pair_cover",)   # whose 95% band is DRAWN. Eight of them is grey
                       # mud, so only the construction we report carries one; the I-mark
                       # in the margin qualifies every curve at once. () for none,
                       # tuple(fk.CRIT) for all
BAND_ALPHA = 0.13
HEADROOM   = 0.26      # fraction of the y range added on the box's side, so a fixed
                       # corner cannot land on the data
BOX_CORNER = "upper left"   # ONE corner in both panels: two boxes in two places read
                       # as two decisions
SHOW_BOX   = True      # the panel's own summary: the best cell, and where ours stands
SHOW_RING  = True      # ring the cell the paper reports
PANEL_LETTERS = True
LETTER_OFFSET = (-22, 1)   # points from the axes' top-left corner: left and up
TITLE_PAD  = 13
TITLE_FONT, TICK_FONT, BOX_FONT = 9.0, 8.0, 7.0
XTICK_FONT, XLABEL_FONT, YLABEL_FONT = 6.5, 8.0, 8.5
LEGEND_NCOL, LEGEND_PAD_IN, LEGEND_FONT = 5, 0.38, 6.8
DS_LABEL   = {"m2or": "M2OR", "cc": "Mosquito (Carey)", "hc": "Fly (Hallem-Carlson)"}
RG_LABEL   = {"transductive": "Transductive", "inductive": "Cold molecule"}
# =============================================================================

df = qg.load(root=ROOT_DIR, dataset=DATASET, regime=REGIME, mol_source=MOL_SOURCE,
             combo=COMBO, split=SPLIT, seeds=SEEDS)

SERIES = sorted(df.series.unique())
QS = sorted(pd.to_numeric(qg.graph_rows(df)["quantile"], errors="coerce").dropna().unique())
CRITS = [c for c in fk.CRIT if c in set(qg.graph_rows(df).criterion)]

In [ ]:
# ---------------------------------------------------------------- what is on disk
# One guard, and it matters: `k_mode` decides what a quantile MEANS. Under
# `coverage_quantile` q cuts on the coverage distribution; under `fraction` it keeps
# the top (1-q) share outright. On a complete matrix the first keeps everything at
# every q. A frame holding both puts two different knobs on one x axis.
modes = sorted(df.k_mode.astype(str).unique())
if len(modes) != 1:
    raise SystemExit(f"this frame holds k_mode {modes} -- two knobs cannot share an "
                     f"axis. Filter to one, or read them as separate figures.")
K_MODE = modes[0]

print(f"{len(SERIES)} series | {len(QS)} quantiles | {len(CRITS)} criteria | "
      f"k_mode: {K_MODE} | split: {SPLIT}")
print("quantiles:", ", ".join(f"{q:g}" for q in QS))
print("files:", ", ".join(pathlib.Path(f).name for f in df.attrs["files"]))

# The second guard: are the seeds the same everywhere? Adding a seed is safe and
# the sweep picks it up, but a run killed part-way through one leaves some cells
# averaging more draws than their neighbours, and the curve then moves by the
# imbalance instead of by the knob. Cut back with SEEDS until this says balanced.
bal = qg.seed_balance(df)
if len(bal) and not bal["balanced"].all():
    print("\nWARNING: unbalanced seeds -- some cells carry fewer than the rest.\n"
          "         Set SEEDS to the seeds that are COMPLETE, or finish the run.")
display(bal)

# `shared` counts rows copied rather than fitted: at q<=0 nothing is cut, so every
# criterion is the same graph there and one fit stands for all of them.
qg.coverage(df)

In [ ]:
MOF = qg.metric_for(df)
DS_OF = dict(zip(df.series, df.dataset))
print("metric of record:", MOF)

# How small a difference this grid can resolve at all: the model noise LEFT IN A FOLD
# MEAN after the seeds are averaged. Every curve below is made of fold means, so a
# wiggle shorter than this is not a finding, whatever the band says. With one seed it
# is undefined -- and then no I-mark is drawn, which is the honest statement.
FLOOR = {}
for s in SERIES:
    f = qg.floor(df[df.series == s], MOF[DS_OF[s]])
    if len(f):
        FLOOR[s] = float(f.iloc[0]["floor"])
if FLOOR:
    display(pd.Series(FLOOR, name="resolution floor").round(4))
else:
    print("one model seed per cell -- no resolution floor; re-run with --seeds 42 43 ...")

---
## 1. The figure

Two panels, M2OR's two regimes. Every curve is the **paired** advantage of one
construction over the boosting base, taken fold by fold, so the base is the zero line
rather than a curve of its own and every interval is the paired one.

The question the panel answers is not "which cell is best" -- with a surface this flat
the argmax is a ranking of noise -- but whether the cell we report is defensible. So
the box names the best cell on the panel, how far the next cell sits behind it in the
best cell's own half-widths, and where ours stands by the same measure. `random` is
the control: a ranked criterion that does not clear it buys message passing, not a
choice of hubs.

In [ ]:
# ========================== THE HEADLINE PAIR ================================
# Eight curves on one axis is the most a colour-vision-safe palette can carry, so each
# criterion has its own MARKER as well as its own hue, and `random` is grey -- the
# control the rest are read against, not a ninth competitor. Only the reported
# construction keeps a band: eight of them is grey mud. The I-mark in the left margin
# is what this grid can resolve at all, and a wiggle shorter than it is not a finding
# whatever the band says.
METRIC_TEX = {"R2": "$R^2$"}
LETTERS = "abcdefghijklmnop"
#: the glyph that matches each criterion's marker, for the box
GLYPH = {"o": "\u25cf", "s": "\u25a0", "^": "\u25b2", "D": "\u25c6", "v": "\u25bc",
         "P": "\u271a", "*": "\u2605", "x": "\u2715"}


def highlight_for(series):
    """Dimming is figure-wide or it is nothing: a key showing eight hues beside a plot
    drawn in grey is worse than no key. When the panels report different criteria,
    nothing is dimmed."""
    if HIGHLIGHT != "auto":
        return HIGHLIGHT
    return qg.PAPER_POINT.get(DS_OF.get(series), (None, None))[0]


_wanted = {highlight_for(s) for s in SERIES}
HL = _wanted.pop() if len(_wanted) == 1 else None
print("highlighted:", HL or "none -- all criteria in their own hue")


def draw_panel(a, ds, regime):
    """One panel: every criterion's paired advantage over the base, against q.

    Returns the panel's summary (best cell, the runner-up behind it, where ours sits),
    or {} if nothing for this cell is on disk."""
    s = f"{ds} / {regime}"
    metric = MOF.get(ds)
    sub = df[df.series == s]
    a.axhline(0.0, color=fk.C["boost_full"], lw=1.3, ls=fk.DASH["boost_full"],
              zorder=2.5)
    if sub.empty or metric is None:
        fk.note_empty(a)
        return {}
    d = qg.delta_vs_boost(sub, metric, level=CI_LEVEL)
    if d.empty:
        fk.note_empty(a)
        return {}
    for crit in CRITS:
        q = d[d.criterion == crit].sort_values("quantile")
        if q.empty:
            continue
        col, mk, lw, z = fk.crit_style(crit, HL)
        if crit in (BAND_CRITS or ()):
            fk.band(a, q["quantile"], q["lo"], q["hi"], col, BAND_ALPHA)
        a.plot(q["quantile"], q["mean"], color=col, marker=mk, lw=lw, zorder=z,
               markersize=4.0)
    fk.knob_axis(a, QS, label="coverage quantile $q$" if K_MODE == "coverage_quantile"
                 else f"quantile ({K_MODE})")
    a.tick_params(axis="y", labelsize=TICK_FONT)
    a.tick_params(axis="x", labelsize=XTICK_FONT)
    a.xaxis.label.set_size(XLABEL_FONT)

    # The ruler goes in a margin of its own, left of q=0: inside the plot it lands on
    # the very curves it is there to qualify.
    x0, x1 = a.get_xlim()
    a.set_xlim(x0 - 0.07, x1)
    fk.resolution_mark(a, FLOOR.get(s), x=x0 - 0.04)

    b = qg.best_cell(d, metric, paper=qg.PAPER_POINT.get(ds))
    if SHOW_RING and b.get("in_grid") and np.isfinite(b.get("ours_mean", np.nan)):
        a.plot([b["ours_q"]], [b["ours_mean"]], marker="o", markersize=11,
               markerfacecolor="none", markeredgecolor=fk.INK, markeredgewidth=1.2,
               zorder=4)
    if HEADROOM:
        y0, y1 = a.get_ylim()
        pad = (y1 - y0) * HEADROOM
        a.set_ylim(y0 - (pad if BOX_CORNER.startswith("lower") else 0),
                   y1 + (pad if BOX_CORNER.startswith("upper") else 0))
    if SHOW_BOX and b:
        # The criterion is named by the leading GLYPH, not spelled out: the longest name
        # is 17 characters and the box has to fit inside a 3.5 in panel.
        lines = [(b["best_criterion"],
                  f"best q={b['best_q']:g}  {b['best_mean']:+.3f}"
                  + (f"  next {b['sep']:.1f} hw" if np.isfinite(b.get("sep", np.nan))
                     else ""))]
        if b.get("in_grid"):
            lines.append((b["ours_criterion"],
                          f"ours q={b['ours_q']:g}  {b['ours_mean']:+.3f}"
                          + (f"  {b['behind']:.1f} hw behind"
                             if np.isfinite(b.get("behind", np.nan)) else "")))
        else:
            lines.append((None, "ours: not in this grid"))
        items = []
        for crit, txt in lines:
            col = fk.CRIT.get(crit, fk.MUTED)
            lead = TextArea(GLYPH.get(fk.CRIT_MARKER.get(crit, "o"), "\u25cf"),
                            textprops=dict(size=BOX_FONT, color=col))
            body = TextArea(txt, textprops=dict(size=BOX_FONT, color=fk.INK))
            items.append(HPacker(children=[lead, body], sep=2.5, pad=0,
                                 align="baseline"))
        box = AnchoredOffsetbox(loc=BOX_CORNER, frameon=True, borderpad=0.28, pad=0.22,
                                child=VPacker(children=items, sep=1.4, pad=0))
        box.patch.set(facecolor="white", edgecolor="#d5d9dd", linewidth=0.6, alpha=0.9)
        a.add_artist(box)
    return b | dict(series=s, metric=metric)


def legend_handles():
    h = []
    for c in CRITS:
        col, mk, lw, _z = fk.crit_style(c, HL)
        h.append(Line2D([], [], color=col, marker=mk, lw=lw, markersize=4.0,
                        label=c + (" (control)" if c == "random" else "")))
    h.append(Line2D([], [], color=fk.C["boost_full"], ls=fk.DASH["boost_full"],
                    label="the base (zero)"))
    if SHOW_RING:
        h.append(Line2D([], [], color=fk.INK, marker="o", ls="none",
                        markerfacecolor="none", markersize=9,
                        label="the cell the paper reports"))
    if BAND_CRITS:
        h.append(Line2D([], [], color=fk.MUTED, lw=6, alpha=BAND_ALPHA * 1.6,
                        label=f"band = {CI_LEVEL:.0%} CI over folds (reported)"))
    return h + [Line2D([], [], color=fk.MUTED, lw=1.2, ls="none", marker="|",
                       markersize=8,
                       label="I = smallest difference this grid resolves")]


w = FIG_WIDTH_IN / len(PANELS)
fig, ax = plt.subplots(1, len(PANELS), squeeze=False,
                       figsize=(w * len(PANELS), w * ASPECT))
PAIR = []
for i, (ds, regime) in enumerate(PANELS):
    a = ax[0][i]
    b = draw_panel(a, ds, regime)
    if b:
        PAIR.append(b)
    mtex = METRIC_TEX.get(MOF.get(ds), MOF.get(ds))
    a.set_title(f"{DS_LABEL.get(ds, ds)} - {RG_LABEL.get(regime, regime)}  ({mtex})",
                fontsize=TITLE_FONT, loc="center", pad=TITLE_PAD)
    if i == 0:
        a.set_ylabel("advantage over base", fontsize=YLABEL_FONT)
    if PANEL_LETTERS:
        a.annotate(f"({LETTERS[i]})", xy=(0, 1), xycoords="axes fraction",
                   xytext=LETTER_OFFSET, textcoords="offset points",
                   fontweight="bold", fontsize=8.5, ha="left", va="bottom",
                   annotation_clip=False)

fk.figlegend(fig, legend_handles(), ncol=LEGEND_NCOL, pad_in=LEGEND_PAD_IN,
             fontsize=LEGEND_FONT, handlelength=1.8, columnspacing=1.1)
fk.savefig(fig, "quantile_pair", FIG_DIR, SAVE_FIGS, formats=FIG_FORMATS)
plt.show()

display(pd.DataFrame(PAIR).round(4))

### 1.2 The absolute levels, as a diagnostic

The panels above show differences, which is the comparison the paper makes. These show
the two levels the difference is taken between, for every series on disk -- the place
to look when a difference moves and it is not obvious which side moved.

In [ ]:
# 1.2  THE ABSOLUTE LEVELS. Every series on disk, not just the pair above: this is the
# diagnostic, and the pair is the figure.
fig, ax = fk.panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.4, h=2.6)
for i, s in enumerate(SERIES):
    a, metric = ax[i], MOF[DS_OF[s]]
    sub = df[df.series == s]
    c = qg.curve(sub, metric, level=CI_LEVEL)
    drawn = False
    for crit in CRITS:
        q = c[c.criterion == crit].sort_values("quantile")
        if q.empty:
            continue
        col, mk, lw, z = fk.crit_style(crit, HL)
        fk.band(a, q["quantile"], q["lo"], q["hi"], col, 0.10 if crit == HL else 0.0)
        a.plot(q["quantile"], q["mean"], color=col, marker=mk, lw=lw, zorder=z,
               markersize=4.5 if crit == HL else 3.2)
        drawn = True
    lv = qg.levels(sub, metric, level=CI_LEVEL)
    if len(lv):
        fk.reference_line(a, lv.iloc[0]["mean"], "boost_full")
    # the cell the paper stands on, ringed rather than annotated: the label would
    # collide with seven other lines on a panel this size
    pp = qg.paper_point(sub, metric, level=CI_LEVEL)
    if len(pp) and bool(pp.iloc[0]["in_grid"]):
        a.plot([pp.iloc[0]["quantile"]], [pp.iloc[0]["mean"]], marker="o",
               markersize=11, markerfacecolor="none", markeredgecolor=fk.INK,
               markeredgewidth=1.2, zorder=4)
    fk.knob_axis(a, QS, label=f"quantile ({K_MODE})")
    if drawn:
        fk.resolution_mark(a, FLOOR.get(s))
    else:
        fk.note_empty(a)
    a.set_title(s, fontsize=8.5)
    a.set_ylabel(metric)

handles = [h for h in legend_handles() if "zero" not in str(h.get_label())]
handles.insert(len(CRITS), Line2D([], [], color=fk.C["boost_full"],
                                  ls=fk.DASH["boost_full"],
                                  label="boost [prot || mol]"))
fk.figlegend(fig, handles, ncol=4, pad_in=0.9, fontsize=LEGEND_FONT,
             handlelength=1.8, columnspacing=1.1)
fk.savefig(fig, "quantile_metric", FIG_DIR, SAVE_FIGS, formats=FIG_FORMATS)
plt.show()

### 1.3 What the quantile actually cuts

K, not q, is the quantity the knob controls. Its own panel and never a second y axis
on the plot above: two quantiles that resolve to the same K are one experiment drawn
twice, and this is where that becomes visible.

In [ ]:
# 1.3  K AGAINST THE QUANTILE.
if SHOW_K:
    k = qg.k_curve(df)
    fig, ax = fk.panels(len(SERIES), ncols=min(3, len(SERIES)), w=3.4, h=2.2)
    for i, s in enumerate(SERIES):
        a = ax[i]
        q = k[k.series == s].sort_values("quantile")
        if q.empty:
            fk.note_empty(a)
        else:
            a.plot(q["quantile"], q["K"], color=fk.MUTED, marker="o")
            fk.band(a, q["quantile"], q["K_min"], q["K_max"], fk.MUTED, 0.18)
            if (q["K"].nunique() == 1):
                a.text(0.5, 0.9, "the knob cuts nothing here", ha="center",
                       transform=a.transAxes, fontsize=8, color=fk.C["fun"])
        fk.knob_axis(a, QS, label=f"quantile ({K_MODE})")
        a.set_title(s, fontsize=8.5)
        a.set_ylabel("molecules kept (K)")
    fig.suptitle("How many train molecules survive the cut", y=1.01, fontsize=11)
    fk.figlegend(fig, [Line2D([], [], color=fk.MUTED, marker="o",
                              label="K (band = fold to fold)")], ncol=1)
    fk.savefig(fig, "quantile_K", FIG_DIR, SAVE_FIGS)
    plt.show()

---
## 2. Where each criterion peaks, and whether the peak is real

`sep` is the gap between the best quantile and the runner-up, measured in half-widths
of the best point's own interval. **Below 1 the optimum is a ranking of noise** -- and
"the knob does not matter over this range" is a result, and a far easier one to defend
than a peak that moves with the seed.

`behind` in the second table is how far the cell the paper reports sits below the best
cell of the same series, in that cell's half-widths.

In [ ]:
for s in SERIES:
    metric = MOF[DS_OF[s]]
    sub = df[df.series == s]
    print(f"\n=== {s}   ({metric}, {K_MODE})")
    display(qg.best_q(sub, metric, level=CI_LEVEL).drop(columns="series").round(4))
    pp = qg.paper_point(sub, metric, level=CI_LEVEL)
    if len(pp):
        display(pp.drop(columns="series").round(4))

---
## 3. The whole battery

A construction that wins on one metric and nowhere else has not won.

In [ ]:
for s in SERIES:
    sub = df[df.series == s]
    metrics = METRICS or qg.metrics_available(sub, dataset=DS_OF[s], which="headline")
    have = [m for m in metrics
            if pd.to_numeric(sub[m], errors="coerce").notna().any()]
    if not have:
        continue
    fig, ax = fk.panels(len(have), ncols=min(4, len(have)), w=2.8, h=2.2)
    for j, m in enumerate(have):
        a = ax[j]
        c = qg.curve(sub, m, level=CI_LEVEL)
        for crit in CRITS:
            q = c[c.criterion == crit].sort_values("quantile")
            if q.empty:
                continue
            col, mk, lw, z = fk.crit_style(crit, HL)
            a.plot(q["quantile"], q["mean"], color=col, marker=mk, lw=lw, zorder=z)
        lv = qg.levels(sub, m, level=CI_LEVEL)
        if len(lv):
            fk.reference_line(a, lv.iloc[0]["mean"], "boost_full")
        fk.knob_axis(a, QS)
        a.set_title(m + ("  (lower is better)" if m in qg.LOWER_IS_BETTER else ""),
                    fontsize=8)
    fig.suptitle(s, y=1.01, fontsize=10.5)
    fk.figlegend(fig, handles[:-1], ncol=4, pad_in=1.0)
    fk.savefig(fig, f"quantile_battery_{s.replace(' / ', '_')}", FIG_DIR, SAVE_FIGS)
    plt.show()

---
## Before quoting a quantile from this page

Reading the best q off these curves and then reporting those same curves chooses the
number and its defence from the same rows. Two honest forms:

* run the sweep with `SPLIT = "val"`, pick there, and read test once;
* or make the weaker claim these figures support directly -- that over the range where
  `sep < 1` the construction does not matter, and the cell we report is not behind the
  best one by more than the grid can resolve.

The second is usually the one worth making. The ablation's job is to show the choice
was not load-bearing, not to crown a winner.